# Claude Certified Architect — Foundations
## Domain 4: Prompt Engineering & Structured Output
**Exam weight: 20%**

This is the most directly testable domain with live code — every task statement maps to a concrete implementation pattern you can observe and verify. The connecting thread is specification precision: explicit criteria, few-shot examples, schema-enforced structure, validation loops, and review architectures are all different approaches to the same fundamental problem of closing the gap between what you described and the output you actually needed.

This domain is heavily API-focused and the most directly testable with live code. Every task statement maps to a concrete implementation pattern.

**Prerequisites:** `pip install anthropic`  
**Auth:** Set `ANTHROPIC_API_KEY` as an environment variable.

### Task Statements Covered
- **4.1** Design prompts with explicit criteria to improve precision and reduce false positives
- **4.2** Apply few-shot prompting to improve output consistency and quality
- **4.3** Enforce structured output using tool use and JSON schemas
- **4.4** Implement validation, retry, and feedback loops for extraction quality
- **4.5** Design efficient batch processing strategies
- **4.6** Design multi-instance and multi-pass review architectures

In [2]:
import anthropic
import json

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
print("Client ready.")

Client ready.


---
## Task Statement 4.1: Design prompts with explicit criteria to improve precision and reduce false positives

A code review pipeline that flags too many false positives quickly becomes useless — developers stop reading it, including the accurate findings. The problem is almost always specification: vague instructions like "check that comments are accurate" give the model nothing to anchor classification against, so it interprets scope differently on every run.

**What this means in practice:** Vague instructions like "check that comments are accurate" produce high false positive rates because the model interprets "accurate" broadly. Explicit criteria define exactly which condition must be true to report an issue. The difference: "flag comments only when the claimed behavior contradicts the actual code behavior" vs "check that comments are accurate." One is a testable condition; the other is a judgment call.

**Why it matters for an architect:** High false positive rates in CI reviews destroy developer trust. If 40% of reported issues are noise, developers start ignoring all of them — including the real ones. Explicit criteria reduce false positives category by category. Temporarily disabling a high-false-positive category while improving its prompt restores trust while you fix it.

**Core concepts:**
- Explicit criteria define exactly which condition must be true to flag an issue — a testable condition, not a judgment call
- Vague instructions ("be conservative", "only report high-confidence findings") do not improve precision; they shift the threshold without increasing specificity
- Concrete code examples for each severity level (one matching case, one non-matching case) anchor classification; abstract definitions alone produce inconsistent results
- High false positive rate in one review category erodes developer trust in all categories — including the accurate ones

**Anti-patterns to avoid:**
- Using adjectives without definitions ("serious issues only", "be thorough") — the model interprets these inconsistently across runs
- Providing severity definitions without concrete code examples — the model applies the definition differently each time
- Keeping a high-false-positive category active in the live review pipeline while investigating it — temporarily disable it to restore trust in the other categories

In [3]:
# Anti-pattern: vague instructions
# Correct pattern: explicit categorical criteria

CODE_SAMPLE = """
def calculate_discount(price: float, user_tier: str) -> float:
    # Apply 10% discount for premium users
    if user_tier == "premium":
        return price * 0.85  # Bug: applies 15% not 10% as comment states
    # Standard users get no discount
    return price

def format_currency(amount: float) -> str:
    # Returns formatted string like '$12.34'
    return f"${amount:.2f}"  # Correct — comment matches behavior

def validate_email(email: str) -> bool:
    # Basic email validation
    return "@" in email  # Acceptable local pattern — '@' check is intentionally basic
"""

VAGUE_PROMPT = """
Review this code. Check that comments are accurate and flag any issues you find.
Be thorough.
"""

EXPLICIT_PROMPT = """
Review this code for the following specific issues ONLY:

1. COMMENT ACCURACY: Flag ONLY when a comment claims a specific behavior 
   that directly contradicts what the code actually does.
   - REPORT: comment says '10% discount' but code applies 15%
   - DO NOT REPORT: comment is imprecise but not contradictory
   - DO NOT REPORT: comment is minimal (e.g., 'basic validation') when code is intentionally simple

2. SECURITY BUGS: Flag only SQL injection, XSS, or hardcoded credentials.
   DO NOT flag general code quality, style, or performance.

For each finding, provide: location, category, specific contradiction or vulnerability.
If nothing meets these exact criteria, output: {"findings": [], "rationale": "<why nothing was flagged>"}
"""

def review_code(prompt_suffix: str, label: str) -> str:
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        messages=[{"role": "user", "content": prompt_suffix + "\n\nCode:\n" + CODE_SAMPLE}]
    )
    return response.content[0].text

print("=== VAGUE prompt ===")
vague_result = review_code(VAGUE_PROMPT, "vague")
print(vague_result)

print("\n" + "="*60)
print("=== EXPLICIT criteria prompt ===")
explicit_result = review_code(EXPLICIT_PROMPT, "explicit")
print(explicit_result)

print("\nOBSERVE:")
print("  Vague prompt: likely flags all three functions (including false positives)")
print("  Explicit prompt: should flag ONLY the calculate_discount comment contradiction")
print("  (15% applied, 10% claimed) — the only true contradiction in the code")

=== VAGUE prompt ===
## Code Review

---

### `calculate_discount`

**🔴 Bug: Comment/code mismatch**

```python
# Apply 10% discount for premium users
if user_tier == "premium":
    return price * 0.85  # Bug: applies 15% not 10% as comment states
```

The inline comment correctly identifies the bug, but **the bug itself is unresolved**. One of the following needs to happen:

- **Fix the code** to match the comment: `return price * 0.90`
- **Fix the comment** to match the code: `# Apply 15% discount for premium users`

A comment labelling something a bug is not a fix — this should be corrected or tracked in an issue, not left as a code comment.

**🟡 No handling of invalid `user_tier` values**

```python
if user_tier == "premium":
    return price * 0.85
return price  # silently treats any unknown tier as "standard"
```

Passing an unrecognised tier (e.g. `"gold"`, `""`, `None`) silently falls through and returns the full price. Depending on context, this could be a silent data error. C

In [4]:
# Explicit severity criteria with concrete code examples
# Vague severity (high/medium/low without definition) produces inconsistent classification.
# Concrete examples anchor the model's judgment.

SEVERITY_CRITERIA_PROMPT = """
Classify the severity of each finding using ONLY these definitions:

CRITICAL: Exploitable security vulnerability or data loss risk in production.
  Example: f"SELECT * FROM users WHERE id = {user_id}" — SQL injection, exploitable now.
  Example: Hardcoded API key in committed code.

HIGH: Bug that causes incorrect behavior in a common scenario.
  Example: discount calculation applies wrong percentage (15% instead of 10%).
  Example: Missing None check on a field that can be null per the schema.

MEDIUM: Bug that causes incorrect behavior only in edge cases or rarely-hit paths.
  Example: Integer overflow only on values > 2^31.
  Example: Race condition that requires precise timing to trigger.

LOW: Code smell or maintainability issue with no correctness impact.
  Example: Function is 200 lines and could be split for readability.
  Example: Variable name 'x' is not descriptive.

DO NOT use your own severity definitions. Use only the above.
"""

TEST_FINDINGS = [
    "SQL query built with f-string: f\"SELECT * FROM orders WHERE id = {order_id}\"",
    "Function applies 15% discount when documentation specifies 10%",
    "Loop variable named 'i' instead of 'index'",
    "Integer overflow possible when processing orders > 2,147,483,647 items"
]

for finding in TEST_FINDINGS:
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        messages=[{"role": "user", "content": f"{SEVERITY_CRITERIA_PROMPT}\n\nFinding: {finding}\n\nSeverity (one word) and one-sentence rationale:"}]
    )
    print(f"Finding: {finding[:60]}...")
    print(f"  Classification: {response.content[0].text.strip()}")
    print()

Finding: SQL query built with f-string: f"SELECT * FROM orders WHERE ...
  Classification: **CRITICAL**: This is a direct SQL injection vulnerability where an attacker can manipulate `order_id` to execute arbitrary SQL commands in production.

Finding: Function applies 15% discount when documentation specifies 1...
  Classification: **HIGH** — The discount calculation applies the wrong percentage (15% instead of the documented 10%), causing incorrect behavior in the common scenario of any discounted transaction.

Finding: Loop variable named 'i' instead of 'index'...
  Classification: **LOW** — Using `i` as a loop variable is a naming/readability concern with no impact on correctness or security.

Finding: Integer overflow possible when processing orders > 2,147,483...
  Classification: **MEDIUM** — This integer overflow only occurs at values exceeding 2^31, which is an extreme edge case unlikely to be hit in normal production usage.



**Key exam facts for 4.1:**
- Explicit criteria define which condition must be true to report — not a judgment call
- Vague instructions (`"be conservative"`, `"only report high-confidence findings"`) do NOT improve precision
- High false positive categories undermine trust in accurate categories
- Fix: temporarily disable high-FP categories while improving their prompt criteria
- Concrete code examples for each severity level produce consistent classification

---
## Task Statement 4.2: Apply few-shot prompting to improve output consistency and quality

When Claude produces inconsistent output across runs — varying formats, different handling of the same edge case — the gap is usually between the rule you described and the examples that would demonstrate it. Two to four well-chosen examples with explicit reasoning for ambiguous cases anchor the model's behavior exactly where instructions alone leave room for interpretation.

**What this means in practice:** Few-shot examples are the most effective technique when detailed instructions alone produce inconsistent results. Two to four targeted examples in the prompt show the model exactly what correct output looks like — including format, handling of ambiguous cases, and the reasoning for choosing one action over another.

**Why it matters for an architect:** Few-shot examples teach generalization, not just pattern matching. A model that sees 3 examples of how to handle ambiguous tool selection can generalize that judgment to novel queries it hasn't seen. Instructions alone describe the rule; examples demonstrate its application at the edge cases where the rule is hard to apply.

**Core concepts:**
- Few-shot examples are the most effective technique when detailed instructions alone produce inconsistent output format or content quality
- 2-4 targeted examples is the effective range; include at least one edge case and one "do not report" example that shows the boundary condition
- Examples enable generalization to novel patterns — the model infers the underlying rule, not just a list of matching inputs
- Including reasoning in examples for ambiguous cases (not just the output) shows *why* a decision was made, enabling correct generalization at the boundary
- `detected_pattern` fields in example outputs enable systematic analysis of which patterns consistently produce false positives

**Anti-patterns to avoid:**
- Providing more than 4-5 examples without clear justification — diminishing returns and later examples may introduce conflicting signals
- Omitting edge cases from the example set — the model applies its own interpretation to cases the examples don't cover
- Examples without reasoning for ambiguous choices — the model learns "what output" but not "why", making it brittle on novel inputs

In [5]:
# Few-shot examples for consistent output format
# Without examples: format varies across runs
# With examples: format is consistent and structured

NO_EXAMPLES_PROMPT = """
Review this code change and report any issues found.
Include location, severity, and a suggested fix.

Code:
def get_user(user_id):
    return db.query(f"SELECT * FROM users WHERE id = {user_id}")
"""

FEW_SHOT_PROMPT = """
Review code changes and report issues in this exact format.

Example 1:
Code: result = cache.get(key) or fetch_from_db(key)
Output:
{
  "location": "cache.py:14",
  "severity": "medium",
  "category": "bug",
  "issue": "Short-circuit evaluation fails when cache returns falsy value (0, empty string, False)",
  "fix": "Use explicit None check: result = cache.get(key); if result is None: result = fetch_from_db(key)",
  "detected_pattern": "falsy_short_circuit"
}

Example 2:
Code: query = \"SELECT * FROM orders WHERE status = '\" + status + \"'\"
Output:
{
  "location": "orders.py:33",
  "severity": "critical",
  "category": "security",
  "issue": "SQL injection via string concatenation of user-controlled 'status' parameter",
  "fix": "Use parameterized query: cursor.execute('SELECT * FROM orders WHERE status = %s', (status,))",
  "detected_pattern": "sql_string_concatenation"
}

Example 3 (acceptable pattern — do not flag):
Code: logger.debug(f"Processing order {order_id}")
Output:
{
  "finding": "none",
  "rationale": "order_id is an internal system identifier, not PII. Debug logging of system IDs is acceptable."
}

Now review this code:
def get_user(user_id):
    return db.query(f"SELECT * FROM users WHERE id = {user_id}")
"""

print("=== Without few-shot examples ===")
no_ex = client.messages.create(model=MODEL, max_tokens=256,
    messages=[{"role": "user", "content": NO_EXAMPLES_PROMPT}])
print(no_ex.content[0].text)

print("\n=== With few-shot examples ===")
with_ex = client.messages.create(model=MODEL, max_tokens=256,
    messages=[{"role": "user", "content": FEW_SHOT_PROMPT}])
print(with_ex.content[0].text)

print("\nOBSERVE:")
print("  Without examples: format varies, detected_pattern field absent, severity may differ")
print("  With examples: consistent JSON structure, detected_pattern included, format matches examples")

=== Without few-shot examples ===
## Code Review Report

### Issue #1: SQL Injection Vulnerability

| Attribute | Detail |
|-----------|--------|
| **Location** | `get_user()`, line 2 |
| **Severity** | 🔴 Critical |
| **Description** | User input is directly interpolated into the SQL query string using an f-string. An attacker could pass a malicious `user_id` (e.g., `1 OR 1=1`) to manipulate the query, expose all user data, or destroy the database. |

**Suggested Fix:** Use parameterized queries:
```python
def get_user(user_id):
    return db.query("SELECT * FROM users WHERE id = ?", (user_id,))
```

---

### Issue #2: No Input Validation

| Attribute | Detail |
|-----------|--------|
| **Location** | `get_user()`, line 1 |
| **Severity** | 🟠 High |
| **Description** | `user_id` is not validated before use. No type check, range check, or

=== With few-shot examples ===
```json
{
  "location": "unnamed_file.py:2",
  "severity": "critical",
  "category": "security",
  "issue": "SQL injec

### Why `detected_pattern` belongs in your few-shot output examples

Look at the few-shot output format in the cell above. Examples 1 and 2 both include a `detected_pattern` field:

```json
{ "detected_pattern": "falsy_short_circuit" }
{ "detected_pattern": "sql_string_concatenation" }
```

Example 3 (the "no finding" case) omits it — there is no pattern to record when nothing was flagged.

**Why it's in the few-shot examples (4.2), not just the schema:**

Including `detected_pattern` in the JSON schema alone means the model *knows the field exists* but treats it as optional and omits it inconsistently. Including it in 2–3 few-shot output examples teaches the model that *every flagged finding should have one*. The field becomes part of the learned output format.

**What it enables downstream (4.4 feedback loop):**

Once the field is reliably present in every finding, you can aggregate dismissals after a few weeks of production use:

```python
from collections import Counter

# findings_log: list of all findings with an 'outcome' field (accepted / dismissed)
dismissed = [f for f in findings_log if f["outcome"] == "dismissed"]
pattern_counts = Counter(f["detected_pattern"] for f in dismissed)

print(pattern_counts.most_common(5))
# [('comment_imprecise_not_contradictory', 47),
#  ('logging_internal_id', 23),
#  ('single_char_variable', 18), ...]
```

The top result (`comment_imprecise_not_contradictory`, dismissed 47 times) is your highest-priority prompt fix — tighten the 4.1 explicit criteria for that category. Without `detected_pattern`, you'd have to read all 47 dismissed findings manually to spot the pattern.

**The sequence across task statements:**

| Task | Role |
|---|---|
| **4.2** | Few-shot examples establish `detected_pattern` as a reliable output field |
| **4.1** | Explicit criteria reduce false positives in the highest-volume categories |
| **4.4** | Feedback loop: aggregate dismissed `detected_pattern` values → identify which category to fix next |

In [6]:
# Few-shot examples for ambiguous-case handling
# Examples demonstrate REASONING for edge cases, not just the output format

AMBIGUOUS_TOOL_SELECTION_PROMPT = """
You are a routing agent. Select the correct tool for each user request.
Available tools: web_search, lookup_customer_record, search_knowledge_base

Examples showing reasoning for ambiguous cases:

Request: "What's the latest Python release?"
Reasoning: This asks for current external information not in our customer system.
Tool: web_search

Request: "What plan is john@example.com on?"
Reasoning: This asks about a specific customer's account data — internal record lookup.
Tool: lookup_customer_record

Request: "How do I reset my password?" 
Reasoning: This is a how-to question about our product features — internal documentation.
Tool: search_knowledge_base

Request: "What's our refund policy?" 
Reasoning: Product policy question — internal documentation, not customer record or web.
Tool: search_knowledge_base

Now select the tool for this request (respond with JSON {"tool": "...", "reasoning": "..."}):
"""

AMBIGUOUS_REQUESTS = [
    "Can you look up order #5512?",
    "What's the current exchange rate for USD to EUR?",
    "How long does shipping usually take?",
    "Is Sarah Johnson's account still active?",
]

print("=== Few-shot: ambiguous tool selection with reasoning ===")
for request in AMBIGUOUS_REQUESTS:
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        messages=[{"role": "user", "content": AMBIGUOUS_TOOL_SELECTION_PROMPT + request}]
    )
    raw = response.content[0].text.strip().replace("```json","").replace("```","").strip()
    try:
        result = json.loads(raw)
        print(f"Request: '{request}'")
        print(f"  Tool:      {result.get('tool')}")
        print(f"  Reasoning: {result.get('reasoning')}")
    except:
        print(f"Request: '{request}' -> {raw[:100]}")
    print()

=== Few-shot: ambiguous tool selection with reasoning ===
Request: 'Can you look up order #5512?'
  Tool:      lookup_customer_record
  Reasoning: The user is asking about a specific order by order number (#5512), which is internal transactional data stored in the customer record system.

Request: 'What's the current exchange rate for USD to EUR?'
  Tool:      web_search
  Reasoning: Exchange rates are real-time financial data that change constantly. This is external information that cannot be found in customer records or internal knowledge base documentation, so a live web search is required to get the current rate.

Request: 'How long does shipping usually take?'
  Tool:      search_knowledge_base
  Reasoning: This is a question about shipping timeframes, which is an internal policy/product information question. The answer would be found in internal documentation, not in a customer record or via external web search.

Request: 'Is Sarah Johnson's account still active?'
  Tool:      lo

In [7]:
# Few-shot examples for extraction from varied document structures
# Different documents encode the same information differently.
# Examples demonstrate how to handle each structural variant.

EXTRACTION_PROMPT = """
Extract the study sample size and primary outcome from medical documents.
Return JSON: {"sample_size": <integer or null>, "primary_outcome": <string or null>, "confidence": "high|medium|low"}

Examples showing different document structures:

Document: "We enrolled 342 participants across three sites. The primary endpoint was 30-day mortality."
Output: {"sample_size": 342, "primary_outcome": "30-day mortality", "confidence": "high"}

Document: "A total of n=89 subjects completed the trial. We measured reduction in systolic blood pressure as our main outcome."
Output: {"sample_size": 89, "primary_outcome": "reduction in systolic blood pressure", "confidence": "high"}

Document: "Approximately two hundred patients were studied. Outcomes included various cardiac events."
Output: {"sample_size": 200, "primary_outcome": "cardiac events", "confidence": "low"}
Note: 'approximately' and vague outcome -> low confidence

Document: "This retrospective analysis examined insurance claims data."
Output: {"sample_size": null, "primary_outcome": null, "confidence": "low"}
Note: No sample size or primary outcome stated — do not invent values.

Now extract from:
"""

TEST_DOCUMENTS = [
    "We randomized 156 patients to receive either treatment A or placebo. The key metric was HbA1c reduction at 12 weeks.",
    "Several dozen subjects participated in the pilot study looking at sleep quality improvements.",
    "Data from our electronic health records system was analyzed for this report."
]

print("=== Few-shot extraction from varied document structures ===")
for doc in TEST_DOCUMENTS:
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        messages=[{"role": "user", "content": EXTRACTION_PROMPT + doc}]
    )
    raw = response.content[0].text.strip().replace("```json","").replace("```","").strip()
    try:
        result = json.loads(raw)
        print(f"Document: '{doc[:60]}...'")
        print(f"  Extracted: {result}")
    except:
        print(f"Document: '{doc[:60]}...' -> {raw}")
    print()

=== Few-shot extraction from varied document structures ===
Document: 'We randomized 156 patients to receive either treatment A or ...' -> {
  "sample_size": 156,
  "primary_outcome": "HbA1c reduction at 12 weeks",
  "confidence": "high"
}


**Reasoning:**
- **Sample size (156):** Explicitly stated as "We randomized 156 patients" — clear, exact integer, no ambiguity.
- **Primary outcome:** "The key metric was HbA1c reduction at 12 weeks" — "key metric" is a direct synonym for primary outcome, and the outcome is specific and measurable

Document: 'Several dozen subjects participated in the pilot study looki...' -> Here's my analysis of the document:

**Reasoning:**
- **Sample size**: "Several dozen" is vague — it implies somewhere in the range of ~24–96, but no specific number is given. I will not invent a precise integer.
- **Primary outcome**: "sleep quality improvements" is mentioned, but without explicit designation as a *primary* outcome (no language like "primary endpoint was" or 

**Key exam facts for 4.2:**
- Few-shot examples are the most effective technique when instructions alone produce inconsistent results
- 2-4 examples is the target range; include at least one edge case and one "do not report" example
- Examples enable generalization to novel patterns — not just matching pre-specified cases
- Include reasoning in examples for ambiguous cases (shows WHY a choice was made)
- Include `detected_pattern` fields to enable systematic analysis of false positive patterns

---
## Task Statement 4.3: Enforce structured output using tool use and JSON schemas

Asking the model to "return JSON" in a prompt is non-deterministic — the response might include markdown fences, a prose preamble, or malformed brackets depending on context. Tool use with a JSON schema eliminates that variability entirely: the API enforces schema compliance at the protocol level, making syntax errors structurally impossible.

**What this means in practice:** Tool use (`tool_use`) with a JSON schema is the most reliable approach for guaranteed schema-compliant output. It eliminates JSON syntax errors because the API enforces schema compliance. However, it does NOT prevent semantic errors (values in wrong fields, totals that don't sum, logically inconsistent values). Schema design matters: use nullable fields for data that may not exist in source documents; use enum + "other" + detail string patterns for extensible categories.

**Why it matters for an architect:** Asking the model to "return JSON" in a prompt is non-deterministic — the model may include markdown fences, add prose, or produce syntax errors. Tool use eliminates all of that. The `tool_choice` distinction matters for pipelines: `"auto"` allows text responses (pipeline failure if you need JSON); `"any"` guarantees a tool call; forced selection guarantees a specific tool.

**Core concepts:**
- `tool_use` with a JSON schema eliminates syntax errors (no markdown fences, no prose preambles, no malformed brackets) — schema compliance is enforced by the API
- `tool_use` does NOT prevent semantic errors (wrong values in correct fields, totals that don't sum, logically inconsistent values) — those require application-level validation logic
- `tool_choice: "auto"` → model may return text (use when tool call is optional); `"any"` → model must call a tool, it picks which one (use when document type is unknown); forced → must call the specific named tool (use to enforce step ordering)
- Nullable fields (`"type": ["string", "null"]`) prevent hallucinated values when source data is absent from the document
- Enum + `"other"` + detail string = extensible categorical fields that don't break when new values appear

**Anti-patterns to avoid:**
- Asking the model to "return JSON" in the prompt — non-deterministic: may include markdown fences, prose preambles, or syntax errors
- Making fields required when the source document may not contain that information — forces the model to hallucinate a value
- Using `tool_choice: "auto"` when the pipeline requires guaranteed structured output — the model may return a text response instead

In [4]:
# Contrast: prompt-based JSON vs tool_use with schema

DOCUMENT = """
Invoice #INV-2024-0891
Date: November 15, 2024
Vendor: Acme Software Inc.

Services:
- Enterprise License (12 months): $8,400.00
- Implementation Services: $2,200.00
- Support Package: $1,200.00

Subtotal: $11,800.00
Tax (8.5%): $1,003.00
Total Due: $12,803.00
Payment Terms: Net 30
"""

# ANTI-PATTERN: ask for JSON in prompt
prompt_json_request = f"""
Extract the invoice data and return it as JSON with fields:
invoice_number, date, vendor, line_items (array), subtotal, tax, total

Document:
{DOCUMENT}
"""

prompt_response = client.messages.create(
    model=MODEL, max_tokens=512,
    messages=[{"role": "user", "content": prompt_json_request}]
)
print("=== ANTI-PATTERN: JSON in prompt ===")
print(prompt_response.content[0].text[:300])
print("\nNote: may include markdown fences, prose, or inconsistent structure")

=== ANTI-PATTERN: JSON in prompt ===
```json
{
  "invoice_number": "INV-2024-0891",
  "date": "November 15, 2024",
  "vendor": "Acme Software Inc.",
  "line_items": [
    {
      "description": "Enterprise License (12 months)",
      "amount": 8400.00
    },
    {
      "description": "Implementation Services",
      "amount": 2200.00


Note: may include markdown fences, prose, or inconsistent structure


### `tool_use` for structured output

You define a tool whose **inputs** are the fields you want extracted. You never execute the tool — you just read the `input` object out of the response. Because the API enforces schema compliance before returning the response, you get guaranteed valid JSON with the right field types.

```python
# Define a tool whose inputs ARE the data fields you want
EXTRACT_TOOL = {
    "name": "extract_invoice",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "total":          {"type": "number"},
            "date":           {"type": "string"}   # YYYY-MM-DD
        },
        "required": ["invoice_number", "total"]
    }
}

response = client.messages.create(
    model=MODEL,
    tools=[EXTRACT_TOOL],
    tool_choice={"type": "tool", "name": "extract_invoice"},  # force the call
    messages=[{"role": "user", "content": f"Extract from: {document}"}]
)

# Pull the data directly — API guaranteed the schema
tool_block = next(b for b in response.content if b.type == "tool_use")
data = tool_block.input
# {"invoice_number": "INV-001", "total": 124.50, "date": "2025-01-15"}
```

Contrast with asking for JSON in the prompt:

```python
# Non-deterministic — may include ```json fences, a prose preamble, or syntax errors
response = client.messages.create(
    messages=[{"role": "user", "content": "Extract the invoice data and return JSON..."}]
)
raw = response.content[0].text  # "Sure! Here's the JSON:\n```json\n{..."
```

**What `tool_use` guarantees vs. what it doesn't:**

| | Eliminated by `tool_use` | Needs application logic |
|---|---|---|
| Markdown fences / prose preamble | ✓ | |
| Missing required fields | ✓ | |
| Wrong field types | ✓ | |
| Values placed in the wrong field | | ✓ |
| Line items that don't sum to subtotal | | ✓ |
| Date that's parseable but incorrect | | ✓ |

The second column — semantic errors — is what task 4.4 handles.

The code cell below defines the full extraction schema used throughout the rest of this section (including the retry loop in 4.4).

In [5]:
# Full invoice extraction tool — schema used by this cell and the 4.4 retry loop below

INVOICE_EXTRACTION_TOOL = {
    "name": "extract_invoice",
    "description": "Extract structured invoice data from a document.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string"},
            "date": {"type": "string", "description": "ISO 8601 date: YYYY-MM-DD"},
            "vendor": {"type": "string"},
            "payment_terms": {
                "type": "string",
                "enum": ["net_30", "net_60", "net_90", "immediate", "other", "unclear"]
            },
            "payment_terms_detail": {
                "type": ["string", "null"],
                "description": "Only populated when payment_terms is 'other'"
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "amount": {"type": "number"}
                    },
                    "required": ["description", "amount"]
                }
            },
            "subtotal": {"type": "number"},
            "tax": {"type": ["number", "null"]},   # nullable: not all invoices have tax
            "total": {"type": "number"},
            "calculated_total": {
                "type": "number",
                "description": "subtotal + tax as calculated from the document"
            },
            "document_currency": {"type": ["string", "null"]},
            "detected_pattern": {
                "type": ["string", "null"],
                "description": (
                    "Short label for the document structure observed "
                    "(e.g. 'standard_itemized', 'date_not_iso', 'tax_embedded'). "
                    "Used to track which patterns cause extraction errors."
                )
            }
        },
        "required": ["invoice_number", "date", "vendor", "line_items", "total"]
        # tax is NOT required — it may be absent from some invoices
    }
}

# Run the extraction — tool_choice forces the specific tool
tool_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[INVOICE_EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[{"role": "user", "content": f"Extract invoice data:\n{DOCUMENT}"}]
)

# Data lives in the tool_use block's .input field — not in text content
tool_use_block = next(b for b in tool_response.content if b.type == "tool_use")
extracted = tool_use_block.input

print("=== Extracted (schema-guaranteed) ===")
print(json.dumps(extracted, indent=2))

# Semantic validation — the schema cannot catch this
stated_total = extracted.get("total", 0)
calculated = extracted.get("calculated_total", 0)
if abs(stated_total - calculated) > 0.01:
    print(f"\nSEMANTIC ERROR: stated_total ({stated_total}) != calculated_total ({calculated})")
    print("Schema enforces types and required fields; value consistency requires application logic.")
else:
    print(f"\nSemantic check passed: stated={stated_total}, calculated={calculated}")

print(f"\ndetected_pattern: '{extracted.get('detected_pattern')}'",)

=== Extracted (schema-guaranteed) ===
{
  "invoice_number": "INV-2024-0891",
  "date": "2024-11-15",
  "vendor": "Acme Software Inc.",
  "line_items": [
    {
      "description": "Enterprise License (12 months)",
      "amount": 8400.0
    },
    {
      "description": "Implementation Services",
      "amount": 2200.0
    },
    {
      "description": "Support Package",
      "amount": 1200.0
    }
  ],
  "subtotal": 11800.0,
  "tax": 1003.0,
  "total": 12803.0,
  "calculated_total": 12803.0,
  "payment_terms": "net_30",
  "detected_pattern": "standard_itemized"
}

Semantic check passed: stated=12803.0, calculated=12803.0

detected_pattern: 'standard_itemized'


In [ ]:
# Demonstrating: enum + "other" + detail string pattern
# The DOCUMENT above has "Net 30" which maps cleanly to the "net_30" enum value.
# This document has an unusual payment term that doesn't match any enum value,
# forcing the model to use "other" and record the actual term in the detail field.

UNUSUAL_TERMS_DOCUMENT = """
Invoice #INV-2024-0892
Date: November 20, 2024
Vendor: SpecialtySupplier Ltd.

Services:
- Consulting (40 hours): $6,000.00

Total Due: $6,000.00
Payment Terms: 2% 10 Net 45
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=[INVOICE_EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[{"role": "user", "content": f"Extract invoice data:\n{UNUSUAL_TERMS_DOCUMENT}"}]
)

tool_block = next(b for b in response.content if b.type == "tool_use")
result = tool_block.input

print("=== Enum + 'other' + detail string ===")
print(f"payment_terms:        {result.get('payment_terms')}")
print(f"payment_terms_detail: {result.get('payment_terms_detail')}")
print()
print("OBSERVE:")
print("  '2% 10 Net 45' doesn't match net_30/net_60/net_90/immediate")
print("  → payment_terms = 'other'  (schema stays valid — no unknown enum value)")
print("  → payment_terms_detail captures the actual term for downstream use")
print()
print("Why this matters: without 'other', an unusual term would either")
print("force a hallucinated enum value or a schema validation error.")


In [ ]:
# Demonstrating: "unclear" enum value for ambiguous cases
# "other" = value is known but non-standard (captured in detail field)
# "unclear" = model cannot determine the value from the source document
#
# Without "unclear", the model must either pick the closest wrong enum value
# or hallucinate something plausible. "unclear" gives it a safe exit.

AMBIGUOUS_TERMS_DOCUMENT = """
Invoice #INV-2024-0893
Date: November 22, 2024
Vendor: GlobalServices Inc.

Services:
- Strategy Consulting: $9,500.00

Total Due: $9,500.00
Payment Terms: As per master agreement
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=[INVOICE_EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[{"role": "user", "content": f"Extract invoice data:\n{AMBIGUOUS_TERMS_DOCUMENT}"}]
)

tool_block = next(b for b in response.content if b.type == "tool_use")
result = tool_block.input

print("=== 'unclear' enum value for ambiguous cases ===")
print(f"payment_terms:        {result.get('payment_terms')}")
print(f"payment_terms_detail: {result.get('payment_terms_detail')}")
print()
print("OBSERVE:")
print("  'As per master agreement' doesn't state actual terms — the model")
print("  cannot determine whether this is net_30, net_60, or anything else.")
print("  → payment_terms = 'unclear'  (safe exit — no hallucinated guess)")
print("  → payment_terms_detail = None  (there's nothing to capture)")
print()
print("Contrast with the 'other' case:")
print("  'other'   = value IS known, just non-standard ('2% 10 Net 45')")
print("  'unclear' = value CANNOT BE DETERMINED from the source document")


In [10]:
# Demonstrating tool_choice options for structured output guarantee

EXTRACTION_TOOL_A = {
    "name": "extract_invoice",
    "description": "Extract invoice data from financial documents (invoices, receipts, purchase orders).",
    "input_schema": {"type": "object",
        "properties": {"invoice_number": {"type": "string"}, "total": {"type": "number"}},
        "required": ["invoice_number", "total"]}
}
EXTRACTION_TOOL_B = {
    "name": "extract_contract",
    "description": "Extract contract data from legal documents (agreements, SOWs, NDAs).",
    "input_schema": {"type": "object",
        "properties": {"parties": {"type": "array", "items": {"type": "string"}},
                       "effective_date": {"type": "string"}},
        "required": ["parties"]}
}

def test_tool_choice(config, label, doc):
    response = client.messages.create(
        model=MODEL, max_tokens=256,
        tools=[EXTRACTION_TOOL_A, EXTRACTION_TOOL_B],
        tool_choice=config,
        messages=[{"role": "user", "content": f"Process this document:\n{doc}"}]
    )
    tools_called = [b.name for b in response.content if b.type == "tool_use"]
    text = [b.text for b in response.content if hasattr(b, "text")]
    print(f"[{label}] tool_choice={config}")
    print(f"  Tools called: {tools_called or 'none (text response)'}")
    if text:
        print(f"  Text: {text[0][:80]}")

SHORT_DOC = "Invoice #001, Total: $100.00"
print("=== tool_choice options ===")
test_tool_choice({"type": "auto"}, "auto", "Hello, can you help me today?")
print()
test_tool_choice({"type": "any"}, "any (must call a tool)", SHORT_DOC)
print()
test_tool_choice({"type": "tool", "name": "extract_invoice"}, "forced: extract_invoice", SHORT_DOC)

print("\nKEY EXAM FACT:")
print("  auto -> model may return text (use when tool call is optional)")
print("  any  -> model must call a tool (use when document type is unknown — model picks correct schema)")
print("  forced -> model must call this specific tool (use to enforce step ordering)")

=== tool_choice options ===
[auto] tool_choice={'type': 'auto'}
  Tools called: none (text response)
  Text: I'd be happy to help! However, the document you've shared doesn't appear to cont

[any (must call a tool)] tool_choice={'type': 'any'}
  Tools called: ['extract_invoice']

[forced: extract_invoice] tool_choice={'type': 'tool', 'name': 'extract_invoice'}
  Tools called: ['extract_invoice']

KEY EXAM FACT:
  auto -> model may return text (use when tool call is optional)
  any  -> model must call a tool (use when document type is unknown — model picks correct schema)
  forced -> model must call this specific tool (use to enforce step ordering)


### Format normalization rules in prompts alongside strict schemas

A JSON schema can enforce that a field is `"type": "number"` — but it cannot tell the model *how to convert* inconsistently formatted source values into that number. That conversion logic must be an explicit instruction in the prompt.

The schema alone is not enough when source documents use varied formatting for the same field:

| Source text | Without normalization rule | With normalization rule |
|---|---|---|
| `$1,200.50` | `1200.5` ✓ | `1200.50` ✓ |
| `1.200,50 EUR` | `1.2005` ✗ (misread as decimal) | `1200.50` ✓ |
| `USD 1200.50` | `1200.5` ✓ | `1200.50` ✓ |
| `1,200.50 dollars` | `1200.5` ✓ | `1200.50` ✓ |

The European format (`1.200,50`) is where the schema alone fails — without an explicit rule, the model may interpret the period as a decimal separator and return `1.2005` instead of `1200.50`.

In [ ]:
# Format normalization rules in prompt alongside strict schema

AMOUNT_TOOL = {
    "name": "extract_amount",
    "description": "Extract the invoice total as a plain decimal number.",
    "input_schema": {
        "type": "object",
        "properties": {
            "total": {"type": "number", "description": "Invoice total as a plain decimal"}
        },
        "required": ["total"]
    }
}

# Four invoices with the same amount expressed in different formats
VARIED_INVOICES = [
    ("US format",       "Invoice Total: $1,200.50"),
    ("European format", "Invoice Total: 1.200,50 EUR"),   # period=thousands, comma=decimal
    ("Code-first",      "Invoice Total: USD 1200.50"),
    ("Written out",     "Invoice Total: 1,200.50 dollars"),
]

NORMALIZATION_RULE = (
    "Extract the invoice total as a plain decimal number (e.g. 1200.50). "
    "Remove all currency symbols, codes, and words. "
    "European-format amounts use a period as the thousands separator and a comma as the "
    "decimal separator — convert these correctly (e.g. '1.200,50' -> 1200.50, not 1.2005)."
)

def extract_total(invoice_text: str, include_normalization_rule: bool) -> float | None:
    system = NORMALIZATION_RULE if include_normalization_rule else "Extract the invoice total."
    response = client.messages.create(
        model=MODEL, max_tokens=128,
        system=system,
        tools=[AMOUNT_TOOL],
        tool_choice={"type": "tool", "name": "extract_amount"},
        messages=[{"role": "user", "content": invoice_text}]
    )
    block = next((b for b in response.content if b.type == "tool_use"), None)
    return block.input.get("total") if block else None

print(f"{'Format':<20} {'Without rule':>15} {'With rule':>12}")
print("-" * 50)
for label, text in VARIED_INVOICES:
    without = extract_total(text, include_normalization_rule=False)
    with_rule = extract_total(text, include_normalization_rule=True)
    correct = 1200.50
    without_ok = "OK" if without == correct else f"WRONG ({without})"
    with_ok    = "OK" if with_rule == correct else f"WRONG ({with_rule})"
    print(f"{label:<20} {str(without_ok):>15} {str(with_ok):>12}")

print()
print("OBSERVE: The schema enforces type=number in both columns.")
print("The European format ('1.200,50') is where the normalization rule makes")
print("the difference — without it, the model may read the period as a decimal")
print("separator and return 1.2005 instead of 1200.50.")


**Key exam facts for 4.3:**
- `tool_use` with JSON schema eliminates syntax errors — it does NOT prevent semantic errors
- `tool_choice: "auto"` → model may return text. `"any"` → must call a tool (model picks). Forced → must call specific tool.
- Nullable fields (`"type": ["string", "null"]`) prevent hallucination when source data is absent
- Enum + `"other"` + detail string = extensible categories
- Extract structured data from the `tool_use` block's `.input` field, not from text content

---
## Task Statement 4.4: Implement validation, retry, and feedback loops for extraction quality

Extraction rarely succeeds perfectly on the first pass — date formats vary, fields get misread, totals don't sum correctly. The difference between a robust pipeline and a fragile one is what happens on failure: retrying without feedback is noise the model can't act on; retrying with the specific error gives it exactly what it needs to self-correct.

**What this means in practice:** When extraction fails validation, append the specific error to the retry prompt — not just "try again." The model uses the error feedback to self-correct. Retries are only effective for format or structural errors; if the required information is absent from the source document, no number of retries will produce it. Tracking `detected_pattern` fields enables systematic analysis of which patterns produce false positives.

**Why it matters for an architect:** Retry-without-feedback is noise. Retry-with-specific-error is signal. The model needs to know *what was wrong* to fix it. "The line items don't sum to the stated subtotal" is actionable. "Please try again" is not.

**Core concepts:**
- Retry-with-error-feedback: each retry appends the original document, the failed extraction, and the specific validation errors — "line items sum to $11,800 but subtotal is $11,900" is actionable; "please try again" is not
- Retries are effective for: format errors (wrong date format), structural errors (wrong field mapping), calculation mistakes the model can recalculate
- Retries are NOT effective when the required information is simply absent from the source document — no number of retries will produce data that isn't there; route to human review instead
- `stated_total` + `calculated_total` + `conflict_detected` boolean enables automated semantic mismatch detection that schema validation cannot provide

**Anti-patterns to avoid:**
- Retrying without including the specific errors in the retry prompt — the model has no signal for what to correct and may reproduce the same failure
- Retrying more than 2-3 times for information that genuinely doesn't exist in the source document
- Treating all extraction failures as retryable — absent information and format errors require different responses

In [11]:
# Validation logic: semantic errors that schema doesn't catch

def validate_invoice_extraction(extracted: dict) -> list:
    """
    Checks semantic validity — things the JSON schema cannot enforce.
    Returns a list of specific error descriptions including detected_pattern labels
    so retries and human reviewers know which document structure caused the failure.
    """
    errors = []

    # Semantic check 1: line items must sum to subtotal
    if "line_items" in extracted and "subtotal" in extracted:
        items_sum = sum(item.get("amount", 0) for item in extracted["line_items"])
        subtotal = extracted.get("subtotal", 0)
        if abs(items_sum - subtotal) > 0.01:
            errors.append(
                f"SEMANTIC ERROR [detected_pattern: 'line_item_sum_mismatch']: "
                f"line_items sum to {items_sum:.2f} but subtotal is {subtotal:.2f}. "
                f"Re-check each line item amount against the source document."
            )

    # Semantic check 2: subtotal + tax should equal total
    if all(k in extracted for k in ["subtotal", "total"]):
        tax = extracted.get("tax") or 0
        expected_total = extracted["subtotal"] + tax
        if abs(expected_total - extracted["total"]) > 0.01:
            errors.append(
                f"SEMANTIC ERROR [detected_pattern: 'total_mismatch']: "
                f"subtotal ({extracted['subtotal']}) + tax ({tax}) = {expected_total:.2f} "
                f"but total is {extracted['total']}. Verify the tax amount and total from the source."
            )

    # Semantic check 3: date format
    import re
    if "date" in extracted and not re.match(r"\d{4}-\d{2}-\d{2}", str(extracted.get("date", ""))):
        errors.append(
            f"FORMAT ERROR [detected_pattern: 'date_not_iso']: "
            f"date '{extracted['date']}' is not ISO 8601 (YYYY-MM-DD). "
            f"Convert it to the correct format."
        )

    return errors


# Test validation with a flawed extraction
FLAWED_EXTRACTION = {
    "invoice_number": "INV-2024-0891",
    "date": "November 15, 2024",  # Wrong format — detected_pattern: date_not_iso
    "vendor": "Acme Software Inc.",
    "line_items": [
        {"description": "Enterprise License", "amount": 8400.00},
        {"description": "Implementation", "amount": 2200.00},
        {"description": "Support", "amount": 1200.00}
    ],
    "subtotal": 11800.00,
    "tax": 1003.00,
    "total": 99999.00,  # Semantic error — detected_pattern: total_mismatch
    "detected_pattern": "date_not_iso"  # Model self-reported the pattern it observed
}

errors = validate_invoice_extraction(FLAWED_EXTRACTION)
print("=== Semantic validation errors (with detected_pattern labels) ===")
for e in errors:
    print(f"  - {e}")
print("\nThe detected_pattern labels enable systematic analysis:")
print("  If 40% of dismissed findings have detected_pattern='date_not_iso',")
print("  that's the highest-value prompt improvement target.")

=== Semantic validation errors (with detected_pattern labels) ===
  - SEMANTIC ERROR [detected_pattern: 'total_mismatch']: subtotal (11800.0) + tax (1003.0) = 12803.00 but total is 99999.0. Verify the tax amount and total from the source.
  - FORMAT ERROR [detected_pattern: 'date_not_iso']: date 'November 15, 2024' is not ISO 8601 (YYYY-MM-DD). Convert it to the correct format.

The detected_pattern labels enable systematic analysis:
  If 40% of dismissed findings have detected_pattern='date_not_iso',
  that's the highest-value prompt improvement target.


In [12]:
# Retry-with-error-feedback loop
# Each retry includes the original document, the failed extraction, and specific errors.

def extract_with_retry(document: str, tool: dict, max_retries: int = 2) -> dict:
    """
    Extraction pipeline with validation-retry loop.
    Each retry appends the specific validation errors to guide self-correction.
    """
    messages = [{"role": "user", "content": f"Extract invoice data:\n{document}"}]

    for attempt in range(max_retries + 1):
        print(f"  Attempt {attempt + 1}/{max_retries + 1}")

        response = client.messages.create(
            model=MODEL, max_tokens=1024,
            tools=[tool],
            tool_choice={"type": "tool", "name": tool["name"]},
            messages=messages
        )

        tool_block = next((b for b in response.content if b.type == "tool_use"), None)
        if not tool_block:
            print("  ERROR: No tool call in response")
            continue

        extracted = tool_block.input
        errors = validate_invoice_extraction(extracted)

        if not errors:
            print(f"  Validation passed on attempt {attempt + 1}")
            return {"success": True, "data": extracted, "attempts": attempt + 1}

        print(f"  Validation failed: {len(errors)} error(s)")
        for e in errors:
            print(f"    - {e}")

        if attempt < max_retries:
            # Append assistant turn (tool call) and feedback as user turn
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [
                    # Return the tool result first
                    {"type": "tool_result", "tool_use_id": tool_block.id,
                     "content": json.dumps(extracted)},
                    # Then provide specific error feedback
                    {"type": "text",
                     "text": f"Validation failed with these specific errors:\n" +
                             "\n".join(f"- {e}" for e in errors) +
                             "\n\nPlease re-extract, correcting only the fields mentioned above. "
                             "Reference the original document to verify the correct values."}
                ]
            })

    return {"success": False, "data": extracted, "attempts": max_retries + 1, "errors": errors}


print("=== Retry-with-error-feedback loop ===")
result = extract_with_retry(DOCUMENT, INVOICE_EXTRACTION_TOOL)
print(f"\nResult: success={result['success']}, attempts={result['attempts']}")
if result["success"]:
    print(f"Extracted total: {result['data'].get('total')}, subtotal: {result['data'].get('subtotal')}")

=== Retry-with-error-feedback loop ===
  Attempt 1/3
  Validation passed on attempt 1

Result: success=True, attempts=1
Extracted total: 12803.0, subtotal: 11800.0


In [13]:
# Demonstrating: when retries are ineffective
# Retries work for: format errors, structural errors, calculation mistakes
# Retries DON'T work for: information absent from the source document

INCOMPLETE_DOCUMENT = """
Service Agreement
Thank you for your business.
Please remit payment at your earliest convenience.
"""
# This document has no invoice number, no amounts, no dates

def attempt_extraction(document: str, tool: dict) -> dict:
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        tools=[tool],
        tool_choice={"type": "tool", "name": tool["name"]},
        messages=[{"role": "user", "content": f"Extract invoice data:\n{document}"}]
    )
    tool_block = next((b for b in response.content if b.type == "tool_use"), None)
    return tool_block.input if tool_block else {}


print("=== Retry effectiveness: retryable vs non-retryable failures ===")
print()
print("Case 1: Information is present but wrongly formatted (RETRYABLE)")
print("  Error: date is 'November 15, 2024' — fix format to '2024-11-15'")
print("  Retry with error feedback: model can correct the format")
print()
print("Case 2: Information is absent from the source document (NOT RETRYABLE)")
extracted = attempt_extraction(INCOMPLETE_DOCUMENT, INVOICE_EXTRACTION_TOOL)
print(f"  Document: '{INCOMPLETE_DOCUMENT.strip()}'")
print(f"  Extracted: {json.dumps(extracted, indent=2)}")
print()
print("  If invoice_number is null, retrying won't produce a value.")
print("  The information doesn't exist in the source. Route to human review.")
print()
print("EXAM KEY: Retries fix format/structure errors. They cannot produce information")
print("that doesn't exist in the source document.")

=== Retry effectiveness: retryable vs non-retryable failures ===

Case 1: Information is present but wrongly formatted (RETRYABLE)
  Error: date is 'November 15, 2024' — fix format to '2024-11-15'
  Retry with error feedback: model can correct the format

Case 2: Information is absent from the source document (NOT RETRYABLE)
  Document: 'Service Agreement
Thank you for your business.
Please remit payment at your earliest convenience.'
  Extracted: {
  "invoice_number": "UNKNOWN",
  "date": "UNKNOWN",
  "vendor": "UNKNOWN",
  "line_items": [],
  "total": 0,
  "detected_pattern": "missing_line_items",
  "payment_terms": "other",
  "payment_terms_detail": "remit payment at your earliest convenience"
}

  If invoice_number is null, retrying won't produce a value.
  The information doesn't exist in the source. Route to human review.

EXAM KEY: Retries fix format/structure errors. They cannot produce information
that doesn't exist in the source document.


**Key exam facts for 4.4:**
- Retry-with-error-feedback: append original document + failed extraction + specific errors (include `detected_pattern` label in error messages)
- Retries are effective for: format errors, structural errors, calculation mistakes
- Retries are NOT effective when: information is simply absent from the source document
- `detected_pattern` field in the extraction schema records which document structure pattern caused each extraction; when reviewers dismiss findings, aggregating `detected_pattern` values reveals which patterns to fix first
- Self-correction: extract `calculated_total` + `stated_total` → flag discrepancies with `conflict_detected` boolean

---
## Task Statement 4.5: Design efficient batch processing strategies

The Message Batches API cuts processing costs in half, but in exchange for latency predictability — results can arrive up to 24 hours later with no SLA guarantee. Choosing the wrong API for a workflow either blocks developers waiting for results or wastes money on synchronous calls that didn't need to be real-time.

**What this means in practice:** The Message Batches API offers 50% cost savings with up to 24-hour processing time and no guaranteed latency SLA. This makes it suitable for latency-tolerant workloads (overnight reports, weekly audits, nightly test generation) and completely unsuitable for blocking workflows (pre-merge checks where developers wait for results). Batch failures are handled by `custom_id`: resubmit only the failed items, not the full batch. Test prompts on a sample before batch-processing large volumes.

**Why it matters for an architect:** Choosing the wrong API for the workflow either blocks developers unnecessarily or wastes money. A pre-merge check that uses the batch API may take 24 hours — making it useless. An overnight report that uses the synchronous API pays double the cost for no benefit.

**Core concepts:**
- Batch API: 50% cost savings, up to 24-hour processing window, no guaranteed latency SLA — cost efficiency in exchange for latency predictability
- `custom_id` is the correlation key that maps each response back to its source request and enables resubmitting only failed items on partial batch failure
- Batch API does not support multi-turn tool calling within a single request — each request is processed independently
- SLA math: `max_submission_delay = SLA_hours − batch_window_hours` → the maximum interval at which batches must be submitted to guarantee the SLA
- Validate prompts on a small sample before committing large volumes to batch processing

**Anti-patterns to avoid:**
- Using the Batch API for blocking developer workflows (pre-merge checks, real-time support routing) — no guaranteed SLA means results may arrive 24 hours later
- Resubmitting the entire batch when only some items fail — use `custom_id` to identify and resubmit only the failed items
- Assuming Batch API supports multi-turn agentic workflows — batch requests are single-turn

### Batch API Workflow Decision Framework

Use the **Batch API** when the workflow is latency-tolerant and high-volume (overnight reports, weekly audits, nightly test generation). Use the **synchronous API** when a developer or customer is actively waiting for results.

| Workflow | Latency Requirement | API to Use | Reason |
|---|---|---|---|
| Pre-merge code review | < 5 min (developer waiting) | Synchronous | Batch API has no guaranteed SLA — could take 24 h. Blocks the developer workflow. |
| Overnight technical debt report | Available by 9 am (runs overnight) | Batch | Non-blocking, latency-tolerant, high volume. 50% cost savings on 500 documents. |
| Weekly compliance audit | Within weekly window (7 days) | Batch | High volume, latency-tolerant. Cost savings significant at this scale. |
| Real-time customer support routing | < 2 s per ticket | Synchronous | Real-time requirement. Batch processing window is incompatible. |
| Nightly test generation | Start of next business day | Batch | Non-blocking; overnight window fits within the 24-hour batch SLA. |

**Key rule:** Use Batch when latency-tolerant and high-volume. Use Synchronous when a developer or customer is waiting.

In [14]:
# Batch API: constructing requests with custom_id for correlation
# custom_id is how you match responses to requests and handle partial failures

SAMPLE_DOCUMENTS = [
    {"id": "contract_001", "text": "Service agreement between Acme Corp and Beta LLC. Effective January 1, 2025. Term: 12 months."},
    {"id": "contract_002", "text": "Software license agreement. Licensor: TechCo. Licensee: StartupXYZ. Annual fee: $50,000."},
    {"id": "contract_003", "text": "Consulting agreement. Parties: GlobalConsult and MidCo. Rate: $250/hour. Duration: 6 months."},
]

def build_batch_requests(documents: list) -> list:
    """
    Constructs batch API requests.
    custom_id is used to correlate responses with source documents
    and to resubmit only failed items on partial batch failure.
    """
    return [
        {
            "custom_id": doc["id"],  # Must be unique; used for response correlation
            "params": {
                "model": MODEL,
                "max_tokens": 256,
                "messages": [{
                    "role": "user",
                    "content": f"Extract: parties involved, effective date, and contract term from this document.\n\n{doc['text']}\n\nReturn JSON only."
                }]
            }
        }
        for doc in documents
    ]

batch_requests = build_batch_requests(SAMPLE_DOCUMENTS)
print("=== Batch request structure (3 documents) ===")
for req in batch_requests:
    print(f"  custom_id: {req['custom_id']}")
    print(f"  prompt: {req['params']['messages'][0]['content'][:80]}...")
    print()

print("NOTE: In production, submit with client.beta.messages.batches.create(requests=batch_requests)")
print("Poll with client.beta.messages.batches.retrieve(batch_id)")
print("Retrieve results with client.beta.messages.batches.results(batch_id)")

=== Batch request structure (3 documents) ===
  custom_id: contract_001
  prompt: Extract: parties involved, effective date, and contract term from this document....

  custom_id: contract_002
  prompt: Extract: parties involved, effective date, and contract term from this document....

  custom_id: contract_003
  prompt: Extract: parties involved, effective date, and contract term from this document....

NOTE: In production, submit with client.beta.messages.batches.create(requests=batch_requests)
Poll with client.beta.messages.batches.retrieve(batch_id)
Retrieve results with client.beta.messages.batches.results(batch_id)


### Real Batch API Pattern

In production, use `client.beta.messages.batches` instead of looping over synchronous calls. The three steps are: submit, poll, retrieve.

**Step 1 — Submit a batch**

```python
batch = client.beta.messages.batches.create(
    requests=[
        {
            "custom_id": doc["id"],          # Unique key used to correlate each response
            "params": {
                "model": MODEL,
                "max_tokens": 256,
                "messages": [{
                    "role": "user",
                    "content": f"Extract parties, effective date, term. Return JSON only.\n\n{doc['text']}"
                }]
            }
        }
        for doc in documents
    ]
)
batch_id = batch.id
```

**Step 2 — Poll until complete**

```python
import time

while True:
    batch_status = client.beta.messages.batches.retrieve(batch_id)
    if batch_status.processing_status == "ended":
        break
    time.sleep(60)   # Check every minute; the window is up to 24 hours
```

> **Teaching simplification:** a blocking `while True / sleep` is shown here for clarity. In a production app you would use a **webhook/callback** — the batch API notifies your endpoint when processing completes, so no process needs to stay alive polling for up to 24 hours.

**Step 3 — Retrieve results and correlate by `custom_id`**

```python
succeeded = []
failed = []

for result in client.beta.messages.batches.results(batch_id):
    if result.result.type == "succeeded":
        succeeded.append({
            "custom_id": result.custom_id,          # Maps back to the source document
            "text": result.result.message.content[0].text
        })
    else:
        failed.append({
            "custom_id": result.custom_id,
            "error": result.result.error.type
        })

# Resubmit ONLY the failed items — not the full batch
if failed:
    retry_ids = {r["custom_id"] for r in failed}
    retry_docs = [d for d in documents if d["id"] in retry_ids]
    # ... build a new batch with retry_docs
```

`custom_id` is the correlation key. On partial failure, identify failed items by `custom_id` and resubmit only those — never re-run the entire batch.

In [15]:
# SLA calculation: batch submission frequency
# Key exam scenario: guarantee a 30-hour SLA with 24-hour batch processing

def calculate_batch_frequency(sla_hours: int, batch_window_hours: int) -> dict:
    """
    Calculates required batch submission frequency to meet SLA.

    If SLA = 30 hours and batch can take 24 hours:
    Max allowed delay between submission and actual submission = 30 - 24 = 6 hours
    So batches must be submitted every 6 hours at most.
    """
    max_submission_delay = sla_hours - batch_window_hours
    if max_submission_delay <= 0:
        return {"feasible": False, "reason": f"Batch window ({batch_window_hours}h) exceeds SLA ({sla_hours}h)"}

    return {
        "feasible": True,
        "submit_every_hours": max_submission_delay,
        "daily_submissions": 24 / max_submission_delay,
        "explanation": f"SLA {sla_hours}h - batch window {batch_window_hours}h = {max_submission_delay}h max delay. Submit every {max_submission_delay}h."
    }

print("=== Batch SLA calculations ===")
scenarios = [
    (30, 24, "30-hour SLA with 24-hour batch"),
    (48, 24, "48-hour SLA with 24-hour batch"),
    (12, 24, "12-hour SLA with 24-hour batch (infeasible)"),
    (72, 24, "72-hour SLA with 24-hour batch"),
]

for sla, window, label in scenarios:
    result = calculate_batch_frequency(sla, window)
    print(f"\n{label}:")
    for k, v in result.items():
        print(f"  {k}: {v}")

=== Batch SLA calculations ===

30-hour SLA with 24-hour batch:
  feasible: True
  submit_every_hours: 6
  daily_submissions: 4.0
  explanation: SLA 30h - batch window 24h = 6h max delay. Submit every 6h.

48-hour SLA with 24-hour batch:
  feasible: True
  submit_every_hours: 24
  daily_submissions: 1.0
  explanation: SLA 48h - batch window 24h = 24h max delay. Submit every 24h.

12-hour SLA with 24-hour batch (infeasible):
  feasible: False
  reason: Batch window (24h) exceeds SLA (12h)

72-hour SLA with 24-hour batch:
  feasible: True
  submit_every_hours: 48
  daily_submissions: 0.5
  explanation: SLA 72h - batch window 24h = 48h max delay. Submit every 48h.


**Key exam facts for 4.5:**
- Batch API: **50% cost savings**, up to **24-hour** processing, **no guaranteed latency SLA**
- Use batch for: overnight reports, weekly audits, nightly test generation (latency-tolerant)
- Do NOT use batch for: blocking pre-merge checks, real-time routing (latency-sensitive)
- Batch API does NOT support multi-turn tool calling within a single request
- `custom_id` correlates requests to responses; resubmit ONLY failed items on partial failure
- SLA math: `max_submission_delay = SLA - batch_window` → submit every N hours to guarantee SLA
- Refine prompts on a sample set before batch-processing large volumes

---
## Task Statement 4.6: Design multi-instance and multi-pass review architectures

Asking Claude to review code it just wrote is structurally compromised: the model retains the reasoning from generation and tends to justify its decisions rather than question them. The solution is architectural — a separate API invocation with no generation context — not a prompt instruction to "be more critical."

**What this means in practice:** A model retains its reasoning context from generation — it's less likely to question decisions it just made. An independent review instance (fresh context, no generation history) catches issues the generator rationalizes away. Multi-pass review splits large reviews into per-file local analysis passes plus a separate cross-file integration pass to avoid attention dilution and contradictory findings.

**Why it matters for an architect:** Self-review is systematically biased. The generator "knows" why it made each decision and tends to justify rather than question those decisions. This is not a model limitation to be prompted away — it's structural. The solution is architectural: two separate invocations with independent contexts.

**Core concepts:**
- Independent review instance: a separate API invocation with no prior generation context catches issues the generating session rationalizes away — this is a structural difference, not a prompting limitation to be prompted around
- Multi-pass architecture: per-file local analysis pass → separate cross-file integration pass; avoids attention dilution (inconsistent findings, omitted files, contradictory observations about the same pattern across files)
- Confidence scores + human routing: calibrate routing thresholds against a labeled validation set; "high confidence" must actually predict "high accuracy" — verify, don't assume

**Anti-patterns to avoid:**
- Instructing the generating session to "review your own code critically" — it retains its generation reasoning and cannot provide an independent perspective on decisions it just made
- Reviewing all files in a single large-context pass — attention dilution produces detailed feedback on some files and superficial or contradictory feedback on others
- Setting confidence routing thresholds based on intuition rather than calibration against labeled data

In [16]:
# Multi-instance review: independent reviewer catches what self-reviewer justifies

CODE_WITH_ISSUES = """
class PaymentProcessor:
    def __init__(self):
        self.api_key = "sk_live_abc123xyz"  # Hardcoded key
        self.retry_count = 3

    def charge(self, amount: float, card_token: str) -> dict:
        # amount comes from user input — no validation
        response = stripe.charge(
            amount=int(amount * 100),  # Assumes amount is float, no type check
            currency="usd",
            source=card_token,
            api_key=self.api_key
        )
        return response
"""

GENERATOR_CONTEXT = """
I wrote this code to process Stripe payments. I used a hardcoded API key because
it's faster for development and I'll move it to env vars before production.
I skipped amount validation because the UI already validates it.
The float conversion works fine for our use case.
"""

def self_review(code: str, generator_context: str) -> str:
    """Review with generator's reasoning context active — biased toward justification."""
    response = client.messages.create(
        model=MODEL, max_tokens=384,
        system=f"You just wrote this code. Your reasoning was: {generator_context} Now review it.",
        messages=[{"role": "user", "content": f"Review for issues:\n{code}"}]
    )
    return response.content[0].text


def independent_review(code: str) -> str:
    """Review with fresh context — no generation history, no reasoning to defend."""
    response = client.messages.create(
        model=MODEL, max_tokens=384,
        system="You are a security-focused code reviewer with no context about how this code was written.",
        messages=[{"role": "user", "content": f"Review for security issues and bugs:\n{code}"}]
    )
    return response.content[0].text


print("=== Self-review (generator's context active) ===")
self_result = self_review(CODE_WITH_ISSUES, GENERATOR_CONTEXT)
print(self_result)

print("\n=== Independent review (fresh context) ===")
independent_result = independent_review(CODE_WITH_ISSUES)
print(independent_result)

print("\nOBSERVE:")
print("  Self-reviewer may accept 'hardcoded key is temporary' as valid reasoning")
print("  Independent reviewer flags it regardless — context is irrelevant to the finding")

=== Self-review (generator's context active) ===
## Code Review

This code has several serious issues that need fixing before any use, including development.

### 1. 🚨 Critical: Live API Key Exposed in Source Code

```python
self.api_key = "sk_live_abc123xyz"  # This is a LIVE key, not test
```

**Problems:**
- Commits to version control expose this permanently, even after deletion
- `sk_live_` prefix means this charges **real money** right now
- Anyone with repo access can make charges

```python
# Fix: use environment variables immediately, not "before production"
import os
self.api_key = os.environ["STRIPE_API_KEY"]
# Fail loudly at startup if missing, rather than discovering it mid-request
```

### 2. 🚨 Critical: No Server-Side Amount Validation

```python
def charge(self, amount: float, card_token: str) -> dict:
    # "UI validates it" is not a security boundary
```

**Problems:**
- HTTP requests can bypass the UI entirely (curl, Burp Suite, etc.)
- Negative amounts, zero charges,

In [3]:
# Multi-pass review: per-file local analysis + cross-file integration pass
# Avoids attention dilution and contradictory findings

CODE_FILES = {
    "src/auth.py": """
def authenticate(token: str) -> dict | None:
    # Returns user dict if valid, None if invalid
    payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
    return payload  # Assumes decode doesn't raise — no try/except
""",
    "src/orders.py": """
def get_order(order_id: str, user_context: dict) -> dict:
    # Assumes user_context is valid (authenticated upstream)
    return db.query("SELECT * FROM orders WHERE id = %s", (order_id,))
    # Note: no check that order belongs to the authenticated user
""",
    "src/api.py": """
def handle_get_order(request):
    token = request.headers.get('Authorization', '').replace('Bearer ', '')
    user = authenticate(token)  # Returns None on failure — not checked
    order = get_order(request.params['order_id'], user)  # user could be None here
    return order
"""
}

def per_file_review_pass(files: dict) -> dict:
    """Pass 1: review each file in isolation for local issues."""
    findings = {}
    for filename, code in files.items():
        print(f"  Reviewing {filename} in isolation...")
        response = client.messages.create(
            model=MODEL, max_tokens=256,
            system="Review this single file for local bugs and security issues. Focus only on issues visible within this file.",
            messages=[{"role": "user", "content": f"File: {filename}\n{code}"}]
        )
        findings[filename] = response.content[0].text
    return findings


def cross_file_integration_pass(files: dict, per_file_findings: dict) -> str:
    """Pass 2: separate pass focused on cross-file trust boundaries and data flow."""
    context = "\n".join(f"=== {k} ===\n{v}" for k, v in files.items())
    findings = "\n".join(f"=== {k} findings ===\n{v}" for k, v in per_file_findings.items())

    response = client.messages.create(
        model=MODEL, max_tokens=384,
        system=(
            "You are reviewing cross-file integration issues: trust boundary violations, "
            "data flow assumptions between modules, inconsistent error propagation across files. "
            "Do NOT repeat issues already identified in the per-file findings."
        ),
        messages=[{"role": "user", "content":
            f"All files:\n{context}\n\nPer-file findings (do not repeat):\n{findings}\n\n"
            "Identify ONLY cross-file issues not already covered."
        }]
    )
    return response.content[0].text


print("=== Multi-pass review architecture ===")
print("\n--- Pass 1: Per-file local analysis ---")
local_findings = per_file_review_pass(CODE_FILES)
for fname, findings in local_findings.items():
    print(f"\n[{fname}]")
    print(findings[:200] + "..." if len(findings) > 200 else findings)

print("\n--- Pass 2: Cross-file integration analysis ---")
integration = cross_file_integration_pass(CODE_FILES, local_findings)
print(integration)

print("\nOBSERVE: Cross-file pass should catch:")
print("  - authenticate() returns None on failure but api.py doesn't check")
print("  - get_order() doesn't verify order ownership — IDOR vulnerability when called from api.py")
print("These span multiple files and are invisible in per-file analysis.")

=== Multi-pass review architecture ===

--- Pass 1: Per-file local analysis ---
  Reviewing src/auth.py in isolation...
  Reviewing src/orders.py in isolation...
  Reviewing src/api.py in isolation...

[src/auth.py]
## Code Review: `src/auth.py`

### Bug: Missing Exception Handling

The most critical issue is that `jwt.decode()` raises exceptions for invalid tokens, but there is **no try/except block** to handle ...

[src/orders.py]
## Security Review: `src/orders.py`

### Critical Issue: Broken Object Level Authorization (BOLA/IDOR)

**Vulnerability: Insecure Direct Object Reference (IDOR)**

The function accepts an `order_id` a...

[src/api.py]
## Code Review: `src/api.py`

### Bug 1: Missing Authentication Check (Security + Logic Error)

The result of `authenticate(token)` is never validated before use. If authentication fails and `None` is...

--- Pass 2: Cross-file integration analysis ---
## Cross-File Integration Issues

### 1. Null Trust Propagation: `authenticate()` Contract M

In [18]:
# Confidence-based review routing
# Model self-reports confidence alongside each finding.
# Low-confidence findings get routed to human review.

CONFIDENCE_REVIEW_PROMPT = """
Review this code and report findings with confidence scores.
For each finding include:
- issue: description
- confidence: 0.0-1.0 (how certain you are this is a real issue, not a false positive)
- severity: critical|high|medium|low

Return JSON: {"findings": [{"issue": "", "confidence": 0.0, "severity": ""}]}

Code:
def process(data):
    # Might be a security issue or might be fine depending on where data comes from
    query = "SELECT * FROM logs WHERE user_id = " + str(data.get('user_id', 0))
    results = db.execute(query)
    # Custom serializer handles None values
    return custom_serialize(results)
"""

conf_response = client.messages.create(
    model=MODEL, max_tokens=384,
    messages=[{"role": "user", "content": CONFIDENCE_REVIEW_PROMPT}]
)

raw = conf_response.content[0].text.strip().replace("```json","").replace("```","").strip()

print("=== Confidence-based review routing ===")
try:
    parsed = json.loads(raw)
    HUMAN_REVIEW_THRESHOLD = 0.7
    for finding in parsed.get("findings", []):
        conf = finding.get("confidence", 0)
        route = "AUTOMATED" if conf >= HUMAN_REVIEW_THRESHOLD else "HUMAN REVIEW"
        print(f"  Finding: {finding.get('issue', '')[:80]}")
        print(f"  Confidence: {conf} | Severity: {finding.get('severity')} | Route: {route}")
        print()
except json.JSONDecodeError:
    print(raw)

=== Confidence-based review routing ===
{
  "findings": [
    {
      "issue": "SQL injection vulnerability: user_id is directly concatenated into the SQL query string without parameterization or sanitization. Even though str() is called, a malicious input could potentially bypass this if data manipulation occurs before this point, and the pattern itself is unsafe and should use parameterized queries.",
      "confidence": 0.95,
      "severity": "critical"
    },
    {
      "issue": "No input validation on user_id: the value from data.get('user_id', 0) is used without type checking or range validation. A non-integer value could cause unexpected behavior or errors.",
      "confidence": 0.8,
      "severity": "high"
    },
    {
      "issue": "No error handling around db.execute(): database errors (connection failure, query error) are not caught, which could cause unhandled exceptions to propagate and potentially expose stack traces or internal details.",
      "confidence": 0.85,
  

**Key exam facts for 4.6:**
- Self-review is biased — the model retains generation reasoning and rationalizes its decisions
- Independent review instance (no prior context) is more effective than self-review instructions or extended thinking
- Multi-pass: per-file local analysis → separate cross-file integration pass (avoids attention dilution and contradictory findings)
- Confidence scores + human review routing: calibrate thresholds using labeled validation sets
- This is the architectural answer to review quality — not a prompting solution

---
## Domain 4 Capstone: Full Structured Data Extraction Pipeline

**Description:** Build an end-to-end extraction pipeline combining all six task statements:
- Explicit review criteria (4.1)
- Few-shot examples for varied document formats (4.2)
- `tool_use` with JSON schema for guaranteed structure (4.3)
- Validation-retry loop with error feedback (4.4)
- Batch processing decision + `custom_id` handling (4.5)
- Independent review instance for quality check (4.6)

In [19]:
# Domain 4 Capstone: Full extraction pipeline

CAPSTONE_TOOL = {
    "name": "extract_contract",
    "description": "Extract structured data from contract documents.",
    "input_schema": {
        "type": "object",
        "properties": {
            "parties": {"type": "array", "items": {"type": "string"},
                        "description": "All parties named in the contract"},
            "effective_date": {"type": ["string", "null"],
                              "description": "ISO 8601 date or null if not stated"},
            "term_months": {"type": ["integer", "null"],
                           "description": "Contract duration in months or null"},
            "contract_type": {
                "type": "string",
                "enum": ["service_agreement", "license", "nda", "consulting", "other"]
            },
            "contract_type_detail": {"type": ["string", "null"]},
            "annual_value": {"type": ["number", "null"],
                            "description": "Annual contract value in USD or null"},
            "detected_document_structure": {
                "type": "string",
                "description": "Brief note on document structure for tracking extraction patterns"
            }
        },
        "required": ["parties", "contract_type"]
    }
}

FEW_SHOT_EXTRACTION_SYSTEM = """
Extract contract data using the extract_contract tool.

Examples of varied document structures:
- Formal: "This Agreement is entered into by and between X and Y, effective January 1, 2025."
  -> parties: ["X", "Y"], effective_date: "2025-01-01"
- Informal: "X will provide services to Y starting next quarter."
  -> parties: ["X", "Y"], effective_date: null (not specified precisely)
- Value embedded: "Annual licensing fee of $75,000" -> annual_value: 75000
- Value absent: No financial terms mentioned -> annual_value: null (do not invent)

Never fabricate values. Use null when information is absent.
"""

TEST_CONTRACTS = [
    {"id": "c001", "text": "Master Services Agreement between CloudTech Inc. and Retailer Corp, effective March 1, 2025, for a term of 24 months. Annual service fee: $120,000."},
    {"id": "c002", "text": "Software license between DevCo and StartupABC. Licensor grants a non-exclusive license. No financial terms specified in this document."},
    {"id": "c003", "text": "Consulting engagement. Consultant will advise Client on technical matters. Duration and compensation to be agreed separately."}
]

print("=== Domain 4 Capstone: Full extraction pipeline ===")

all_results = []
for contract in TEST_CONTRACTS:
    print(f"\nProcessing {contract['id']}...")

    # Extract with tool_use (4.3)
    response = client.messages.create(
        model=MODEL, max_tokens=512,
        system=FEW_SHOT_EXTRACTION_SYSTEM,  # Few-shot examples (4.2)
        tools=[CAPSTONE_TOOL],
        tool_choice={"type": "tool", "name": "extract_contract"},
        messages=[{"role": "user", "content": contract["text"]}]
    )

    tool_block = next((b for b in response.content if b.type == "tool_use"), None)
    if tool_block:
        extracted = tool_block.input
        # Validate (4.4)
        issues = []
        if not extracted.get("parties"):
            issues.append("No parties extracted")
        if extracted.get("annual_value") is not None and extracted["annual_value"] < 0:
            issues.append("Negative annual_value")

        result = {
            "custom_id": contract["id"],  # For batch correlation (4.5)
            "data": extracted,
            "valid": len(issues) == 0,
            "validation_issues": issues
        }
        all_results.append(result)
        status = "VALID" if result["valid"] else f"INVALID: {issues}"
        print(f"  {status}")
        print(f"  Parties: {extracted.get('parties')}")
        print(f"  Type: {extracted.get('contract_type')} | Value: {extracted.get('annual_value')} | Term: {extracted.get('term_months')}mo")

# Summary
valid = sum(1 for r in all_results if r["valid"])
print(f"\nPipeline complete: {valid}/{len(all_results)} valid extractions")
print(f"Failed: {[r['custom_id'] for r in all_results if not r['valid']]}")
print("Failed items would be resubmitted by custom_id with validation error feedback.")

=== Domain 4 Capstone: Full extraction pipeline ===

Processing c001...
  VALID
  Parties: ['CloudTech Inc.', 'Retailer Corp']
  Type: service_agreement | Value: 120000 | Term: 24mo

Processing c002...
  VALID
  Parties: ['DevCo', 'StartupABC']
  Type: license | Value: None | Term: Nonemo

Processing c003...
  VALID
  Parties: ['Consultant', 'Client']
  Type: consulting | Value: None | Term: Nonemo

Pipeline complete: 3/3 valid extractions
Failed: []
Failed items would be resubmitted by custom_id with validation error feedback.


---
## Domain 4 Complete

**Summary of key exam facts:**

| Task | Core Principle |
|------|----------------|
| 4.1 | Explicit criteria = testable conditions, not judgment calls. Vague instructions don't improve precision. High-FP categories erode trust in accurate ones. |
| 4.2 | 2-4 targeted examples beat detailed instructions for consistency. Include edge cases and reasoning for ambiguous choices. `detected_pattern` enables FP analysis. |
| 4.3 | `tool_use` + schema eliminates syntax errors but NOT semantic errors. `"any"` guarantees a tool call. Nullable fields prevent hallucination. |
| 4.4 | Retry with specific error feedback. Retries fix format/structure errors. Cannot produce absent information. |
| 4.5 | Batch = 50% cost, 24h window, no SLA. Never for blocking workflows. `custom_id` for correlation + partial resubmission. |
| 4.6 | Independent instance beats self-review. Multi-pass: per-file local + cross-file integration. Confidence scores route to human review. |